In [1]:
import torch
import functools
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer, DPOTrainer
from peft import prepare_model_for_kbit_training
from peft import LoraConfig
from huggingface_hub import login
from dotenv import load_dotenv
import os
from pipelineMethods import TrainPipeline

c:\Users\stefv\anaconda3\envs\train_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# torch.load = functools.partial(torch.load, weights_only=False)

trainPipeline = TrainPipeline()

In [3]:
# Go zema tokenot od .env 
load_dotenv() 
HF_HUB_TOKEN = os.getenv("HF_HUB_TOKEN")
trainPipeline.hf_login(HF_HUB_TOKEN)

In [4]:
# dataset_path = r"allenai-tulu-3-sft-personas-instruction-following-mk.jsonl"
dataset_path = r"argilla-ifeval-like-data-fixed-mk.jsonl"
# dataset_path = r"./data/train.jsonl"

ds = trainPipeline.get_dataset(dataset_path)

In [5]:
ds

Dataset({
    features: ['prompt', 'response', 'key', 'prompt1', 'response1', 'instruction_id_list', 'kwargs', 'prompt_level_strict_acc', 'inst_level_strict_acc', 'prompt_level_loose_acc', 'inst_level_loose_acc'],
    num_rows: 56339
})

In [6]:
data = ds.select_columns(["prompt", "response"])
data[:2]

{'prompt': ['Вашиот одговор треба да содржи најмалку 3 реченици. Означете најмалку 2 дела во вашиот одговор со markdown, т.е. *означен дел*. Целиот ваш одговор треба да биде на англиски јазик и со мали букви. Не се дозволени големи букви. Во вашиот одговор, зборот „писмо“ треба да се појави најмалку 3 пати.',
  'Вашиот одговор треба да содржи најмалку 3 реченици. Вклучете го клучниот збор „иновација“ во вашиот одговор. Завршете го одговорот со токму оваа фраза: „Со нетрпение ги очекувам вашите повратни информации.“'],
 'response': ['„Би рекол дека *првиот означен дел* ја воведува темата за мали букви и важноста од почитување на ограничувањето за употреба без големи букви. *вториот означен дел* се навлегува во барањето за фреквенција за зборот буква, осигурувајќи се дека се појавува најмалку три пати во одговорот. Овој пристап помага во одржувањето на структурата и јасноста на одговорот, а воедно ги исполнува сите наведени ограничувања.“',
  'Иновацијата го поттикнува напредокот и ни по

In [ ]:
dataset = data.map(trainPipeline.preprocess_function_sft)

In [ ]:
# split_dataset = data.train_test_split(test_size=0.1, seed=42)

# train_data = split_dataset['train']
# test_data = split_dataset['test']

# train_data.to_json("./data/argilla-ifeval-like-data-fixed-mk-train.jsonl", force_ascii=False)
# test_data.to_json("./data/argilla-ifeval-like-data-fixed-mk-test.jsonl", force_ascii=False)

In [4]:
dataset_path = r"./data/argilla-ifeval-like-data-fixed-mk-train.jsonl"


dataset = trainPipeline.get_dataset(dataset_path)

In [8]:
dataset[:3]

{'prompt': ['Можете ли да дадете список од 4 предмети што се неопходни за пикник во паркот? Вашиот одговор треба да содржи точно 4 точки. Користете ги точките за оценка како што се: * Ова е точка 1. Целиот ваш одговор треба да биде на англиски јазик и со мали букви. Не се дозволени големи букви.',
  'Дајте детално објаснување за процесот на обука на невронска мрежа, вклучувајќи ги клучните чекори и компоненти вклучени. Вашиот одговор мора да содржи наслов, завиткан во двојни аголни загради, како на пример <<песна на радоста>>. Користете ги следните делови: <<Вовед>>, <<Клучни компоненти>>, <<Процес на обука>> и <<Заклучок>>. Означете го почетокот на секој дел со ДЕЛ X, како на пример: ДЕЛ 1. ДЕЛ 1 треба да биде Вовед, ДЕЛ 2 треба да биде Клучни компоненти, ДЕЛ 3 треба да биде Процес на обука, а ДЕЛ 4 треба да биде Заклучок. Осигурајте се дека буквата „e“ се појавува најмалку 10 пати во вашиот одговор.',
  'Ми треба детално објаснување за тоа како функционира енкрипцијата, но осигурајте

In [9]:
len(dataset)

50705

In [10]:
dataset

Dataset({
    features: ['prompt', 'response'],
    num_rows: 50705
})

In [5]:
dataset = dataset.map(trainPipeline.preprocess_function_sft)

In [6]:
dataset = dataset.select_columns(['messages'])

In [7]:
dataset

Dataset({
    features: ['messages'],
    num_rows: 50705
})

In [8]:
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Free VRAM:  {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total VRAM: 4.3 GB
Free VRAM:  3.5 GB


In [9]:
model_name = "google/gemma-3-1b-it"
# model_name = "google/gemma-3-4b-it"

model, tokenizer, lora, training_args = trainPipeline.prepare_model(model_name, HF_HUB_TOKEN)

In [ ]:
# first_row_ids = tokenized_dataset[0]["input_ids"]

# # 2. Decode it back to text
# # clean_up_tokenization_spaces=False preserves the exact structure
# decoded_text = tokenizer.decode(first_row_ids, skip_special_tokens=False)

# print("--- DECODED FORMAT ---")
# print(decoded_text)
# print("----------------------")

In [ ]:
# dataset_final = tokenized_dataset

In [ ]:
# dataset_final

In [ ]:

# train_size = int(1 * len(dataset_final))
# train_split = dataset_final.select(range(0, train_size))
# # test_split  = dataset_final.select(range(train_size, len(dataset_final)))

# # print(f"Train: {len(train_split)}, Test: {len(test_split)}")
# print(f"Sample keys: {train_split[0].keys()}")
# print(f"input_ids length: {len(train_split[0]['input_ids'])}")



In [10]:
print(dataset[0])

{'messages': [{'content': 'Можете ли да дадете список од 4 предмети што се неопходни за пикник во паркот? Вашиот одговор треба да содржи точно 4 точки. Користете ги точките за оценка како што се: * Ова е точка 1. Целиот ваш одговор треба да биде на англиски јазик и со мали букви. Не се дозволени големи букви.', 'role': 'user'}, {'content': '* ќебе за пикник за седење\n* ладилник со пијалоци и закуски\n* крема за сончање за заштита од сонце\n* фризби за забавни игри', 'role': 'assistant'}]}


In [ ]:
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM


def formatting_prompts_func(example):
    return [tokenizer.apply_chat_template(msg, tokenize=False) for msg in example['messages']]


collator = DataCollatorForCompletionOnlyLM(
    response_template="<start_of_turn>model\n", 
    tokenizer=tokenizer
)


trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    data_collator=collator,
    dataset_text_field=None,  
    formatting_func=formatting_prompts_func, # Use this instead of renaming columns
    peft_config=lora,
    max_seq_length=1024,
)
# trainer.train()

c:\Users\stefv\anaconda3\envs\train_env\lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
c:\Users\stefv\anaconda3\envs\train_env\lib\site-packages\trl\trainer\sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [17]:
trainer.train()

Step,Training Loss
10,3.088100
20,3.040500
30,3.037400
40,2.903600
50,2.683800
60,2.400000
70,2.309600
80,2.217200
90,2.176700
100,2.059800


c:\Users\stefv\anaconda3\envs\train_env\lib\site-packages\trl\trainer\utils.py:160: UserWarning: Could not find response key `<start_of_turn>model
` in the following instance: <bos><bos><start_of_turn>user
Дајте детално објаснување за влијанието на уништувањето на шумите врз животната средина. Вашиот одговор треба да содржи најмалку 3 пасуси одделени со разделувачи на ознаки. Вклучете клучни зборови: [уништување на шумите], [биодиверзитет], [стакленички гасови], [јаглерод диоксид], [ерозија на почвата]. Осигурајте се дека зборот [животната средина] се појавува најмалку 2 пати. Означете најмалку 2 дела со ознаки.
***

Уништувањето на шумите, големото отстранување на пошумените површини, има длабоки влијанија врз *животната средина* и планетата како целина. Еден од најзначајните ефекти е губењето на биодиверзитетот. Шумите се дом на широк спектар на растителни и животински видови, од кои многу не се наоѓаат никаде на друго место на Земјата. Кога овие живеалишта се уништени, безброј видов

KeyboardInterrupt: 

# Continue Training Model From Checkpoint

In [12]:
trainer.train(resume_from_checkpoint=True)

Step,Training Loss
1620,1.023800
1630,1.018600
1640,0.980100
1650,1.066700
1660,0.981300
1670,1.039700
1680,0.993000
1690,1.094900
1700,1.000100
1710,1.050300


c:\Users\stefv\anaconda3\envs\train_env\lib\site-packages\trl\trainer\utils.py:160: UserWarning: Could not find response key `<start_of_turn>model
` in the following instance: <bos><bos><start_of_turn>user
Вашиот одговор треба да содржи најмалку 200 зборови. Вашиот одговор мора да има 4 пасуси. Пасусите се одделени со ознака за ознаки: ***

Вашиот одговор мора да содржи наслов, завиткан во двојни аголни загради, како на пример <<Разбирање на основите на машинското учење>>.

Вашиот одговор мора да содржи постскриптум што започнува со PS

Целиот ваш одговор треба да биде на англиски јазик и со сите мали букви. Не се дозволени големи букви.

Во вашиот одговор, зборот „алгоритам“ треба да се појави најмалку 5 пати.

Во вашиот одговор, буквата „l“ треба да се појави најмалку 10 пати.

Вашиот одговор треба да ги објасни основите на машинското учење на едноставен и разбирлив начин за почетници.

<<разбирање на основите на машинското учење>>

Машинското учење е подмножество на вештачката интел

TrainOutput(global_step=3169, training_loss=0.47867698254138374, metrics={'train_runtime': 28353.0713, 'train_samples_per_second': 1.788, 'train_steps_per_second': 0.112, 'total_flos': 8.71325621497198e+16, 'train_loss': 0.47867698254138374, 'epoch': 0.9999802780790848})

# Evaliation stage

In [26]:
dataset_path = r"./data/argilla-ifeval-like-data-fixed-mk-test.jsonl"


dataset_test = trainPipeline.get_dataset(dataset_path)

In [27]:
dataset_test = dataset_test.select(range(3))

In [28]:
dataset_test

Dataset({
    features: ['prompt', 'response'],
    num_rows: 3
})

In [29]:
dataset_test = dataset_test.map(trainPipeline.preprocess_function_sft)

In [30]:
dataset_test = dataset_test.select_columns(['messages'])

In [31]:
dataset_test

Dataset({
    features: ['messages'],
    num_rows: 3
})

In [ ]:
import torch

def evaluate_model_on_test(model, tokenizer, dataset, num_samples=3):
    model.eval()
    
    num_samples = min(num_samples, len(dataset))
    print(f"--- Evaluating {num_samples} Samples ---\n")

    for i in range(num_samples):
        messages = dataset[i]['messages']
        
        prompt_messages = [m for m in messages if m['role'] != 'assistant']
        ground_truth = next((m['content'] for m in reversed(messages) if m['role'] == 'assistant'), "No GT found")

       
        input_ids = tokenizer.apply_chat_template(
            prompt_messages, 
            tokenize=True, 
            add_generation_prompt=True, 
            return_tensors="pt"
        ).to(model.device) 
        
        
        with torch.no_grad():
            
            device_type = "cuda" if torch.cuda.is_available() else "cpu"
            with torch.autocast(device_type=device_type, dtype=model.dtype):
                output_ids = model.generate(
                    input_ids=input_ids,
                    max_new_tokens=512,
                    do_sample=True,
                    temperature=0.2,
                    top_p=0.9,
                    repetition_penalty=1.2,
                    eos_token_id=tokenizer.eos_token_id,
                )

        generated_tokens = output_ids[0][input_ids.shape[1]:]
        prediction_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        print(f"SAMPLE {i+1}")
        print(f"PROMPT:\n{messages[0]['content'][:500]}")
        print("-" * 30)
        print(f"EXPECTED:\n{ground_truth[:500]}")
        print("-" * 30)
        print(f"GENERATED:\n{prediction_text}")
        print("\n" + "="*50 + "\n")


evaluate_model_on_test(model, tokenizer, dataset_test)

--- Evaluating 3 Samples ---

SAMPLE 1
PROMPT:
Во контекст на деловен состанок, дајте совети за ефикасна комуникација. Вашиот одговор мора да содржи наслов, завиткан во двојни аголни загради, како на пример <<песна на радоста>>. Вашиот одговор треба да содржи најмалку 5 реченици. Означете најмалку 2 дела во вашиот одговор со markdown, т.е. *означен дел*. На крајот од вашиот одговор, ве молиме експлицитно додадете постскриптум што започнува со PS
------------------------------
EXPECTED:
<<Совети за ефективна комуникација на деловни состаноци>>

Ефективната комуникација на деловните состаноци е клучна за постигнување на посакуваните резултати и одржување на професионална средина. *Прво, важно е да бидете јасни и концизни во вашата комуникација.* Ова значи директно да се премине на поентата и да се избегнува непотребен жаргон или сложен јазик што може да ги збуни учесниците. *Второ, активното слушање е клучно.* Обрнете внимание на она што го кажуваат другите и покажете дека слуша
--------